In [ ]:
import scipy.io as sio
import matplotlib.pyplot as plt
from tqdm import tqdm
import os, glob, torch, numpy as np, json
import torch.nn as nn
DATA_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', 'data'))
test_split_path = os.path.join(DATA_ROOT, 'processed', 'test_files.json')
if not os.path.exists(test_split_path):
    raise RuntimeError(f'No test split found at {test_split_path}. Run Notebook 03 first.')
with open(test_split_path, 'r') as f:
    mat_files = json.load(f)
print(f'Loaded {len(mat_files)} hold-out files.')


In [ ]:
# ── Configuration & Paths ─────────────────────────────────────────────────────
HPARAMS = {
    'patch_size': 64,
    'n_frames': 8,
    'n_classes': 3
}
FINAL_CKPT = os.path.join(DATA_ROOT, 'processed', 'cnn_lstm_final.pth')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PATCH_SIZE = HPARAMS['patch_size']
N_FRAMES   = HPARAMS['n_frames']
CLASS_NAMES = ['Benign', 'Pre-clinical', 'Malignant']


In [ ]:
class UNetEncoder(nn.Module):
    def __init__(self, channels=[1, 16, 32, 64, 128]):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(channels[0], channels[1], 3, padding=1), nn.ReLU(), nn.BatchNorm2d(channels[1]))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(channels[1], channels[2], 3, padding=1), nn.ReLU(), nn.BatchNorm2d(channels[2]))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(channels[2], channels[3], 3, padding=1), nn.ReLU(), nn.BatchNorm2d(channels[3]))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), nn.Conv2d(channels[3], channels[4], 3, padding=1), nn.ReLU(), nn.BatchNorm2d(channels[4]))
        self.pool = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        x = self.enc1(x)
        x = self.enc2(x)
        x = self.enc3(x)
        x = self.enc4(x)
        return self.pool(x).view(x.size(0), -1)

class CNNLSTMElastography(nn.Module):
    def __init__(self, cnn_channels=[1, 16, 32, 64, 128], lstm_hidden=256, lstm_layers=2, n_classes=3, patch_size=64):
        super().__init__()
        self.cnn = UNetEncoder(channels=cnn_channels)
        self.lstm = nn.LSTM(cnn_channels[-1], lstm_hidden, lstm_layers, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(nn.Linear(lstm_hidden*2, 128), nn.ReLU(), nn.Linear(128, n_classes))
        self.disp_head = nn.Sequential(nn.Linear(lstm_hidden*2, 256), nn.ReLU(), nn.Linear(256, patch_size*patch_size))
    def forward(self, x):
        B, T, C, H, W = x.shape
        x_flat = x.view(B*T, C, H, W)
        feats = self.cnn(x_flat).view(B, T, -1)
        lstm_out, (h_n, _) = self.lstm(feats)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=-1)
        logits = self.classifier(h_final)
        disp = self.disp_head(lstm_out[:, -1, :]).view(B, H, W)
        return logits, disp.unsqueeze(1)


In [ ]:
def prepare_rf_tensor(rf_clean, patch_size=64, n_frames=8, device='cpu'):
    """Extract central spatial patch and first n_frames, return (1,T,1,H,W) tensor."""
    D, W, F = rf_clean.shape
    d0 = max(0, (D - patch_size) // 2)
    w0 = max(0, (W - patch_size) // 2)
    patch = rf_clean[d0:d0+patch_size, w0:w0+patch_size, :]
    if patch.shape[0] < patch_size or patch.shape[1] < patch_size:
        patch = np.pad(patch, ((0, max(0, patch_size-patch.shape[0])), (0, max(0, patch_size-patch.shape[1])), (0,0)), mode='edge')
    t_end = min(n_frames, F)
    patch_t = patch[:, :, :t_end]
    if patch_t.shape[2] < n_frames:
        patch_t = np.pad(patch_t, ((0,0),(0,0),(0, n_frames-patch_t.shape[2])), mode='edge')
    vmax = np.abs(patch_t).max()
    if vmax > 0: patch_t = patch_t / vmax
    t = torch.from_numpy(patch_t).float().permute(2,0,1).unsqueeze(1).unsqueeze(0)
    return t.to(device)

def compute_cnr(displacement: np.ndarray, target_roi: tuple = None) -> float:
    """Compute Elastographic Contrast-to-Noise Ratio (CNR)."""
    H, W = displacement.shape
    if target_roi is None:
        r0, r1, c0, c1 = H // 4, 3 * H // 4, W // 4, 3 * W // 4
    else: r0, r1, c0, c1 = target_roi
    target_vals = displacement[r0:r1, c0:c1].ravel()
    mask = np.ones((H, W), dtype=bool); mask[r0:r1, c0:c1] = False
    bg_vals = displacement[mask].ravel()
    mu_t, mu_b = target_vals.mean(), bg_vals.mean()
    var_t, var_b = target_vals.var(ddof=1), bg_vals.var(ddof=1)
    return abs(mu_t - mu_b) / np.sqrt(var_t + var_b + 1e-8)

def baseline_ncc_cnr(rf_pre: np.ndarray, rf_post: np.ndarray) -> float:
    """Classical 1-D NCC baseline."""
    D, W = rf_pre.shape
    win, step = 32, 16
    depths = range(0, D - win, step)
    disp_map = np.zeros((len(depths), W))
    for j, d0 in enumerate(depths):
        for c in range(W):
            s1, s2 = rf_pre[d0:d0+win, c], rf_post[d0:d0+win, c]
            corr = np.correlate((s1-s1.mean())/(s1.std()+1e-8), (s2-s2.mean())/(s2.std()+1e-8), mode='full')
            disp_map[j, c] = np.argmax(corr) - (win - 1)
    return compute_cnr(disp_map)


In [ ]:
# ── Instantiate & Load Model ──────────────────────────────────────────────────
model = CNNLSTMElastography(n_classes=3, patch_size=64).to(DEVICE)
if os.path.exists(FINAL_CKPT):
    checkpoint = torch.load(FINAL_CKPT, map_location=DEVICE)
    model.load_state_dict(checkpoint.get('state_dict', checkpoint))
    model.eval()
    print(f'✅ Loaded final weights from {FINAL_CKPT}')
else:
    print(f'⚠️ Warning: {FINAL_CKPT} not found. Using random weights.')


# 📊 Phase 3: Clinical Robustness & Statistical Validation

**Dataset Used:** Dataset 2 — `ultrasound-elastography-dataset-for-unsupervised-training`  
**(Hold-out test split — 20% of phantom/in-vivo data, NEVER seen during training)**

**Objective:** Prove statistical viability of the trained CNN-LSTM on unseen physical phantom `.mat` files.

**Pipeline:**
1. Load fully-trained model (`cnn_lstm_final.pth`) in `model.eval()` mode
2. Run Phase-1 DMD pre-processing on hold-out phantom `.mat` files
3. Compute **Elastographic Contrast-to-Noise Ratio (CNR)** per sample
4. **MLE & 95% Confidence Intervals** on mean CNR
5. **Wilcoxon signed-rank hypothesis test** vs. baseline NCC algorithm
6. Classification performance: confusion matrix, per-class precision/recall/F1
7. Visualise strain heatmaps on representative phantom scans

> **Integrates Syllabus Unit 3:** Statistical Estimation Theory (MLE, Confidence Intervals), Hypothesis Testing (Wilcoxon signed-rank, p-value)


## 0. Environment Setup & Imports

In [ ]:
# Redundant or synthetic generation removed by standardization script


## 1. Re-instantiate Architecture & Load Final Weights

In [ ]:
def extract_rf_frames(fpath: str) -> np.ndarray:
    """
    Load .mat file and extract the RF signal matrix.
    Returns array of shape (depth, width, n_frames) with float64.
    """
    import scipy.io as sio
    mat_data = sio.loadmat(fpath)
    for key in ['rf_pre', 'rf_post', 'RF', 'rfData', 'IQpre', 'IQpost', 'Data', 'data', 'rf']:
        if key in mat_data:
            arr = np.array(mat_data[key], dtype=np.float64)
            # Handle (n_frames, depth, width) -> (depth, width, n_frames)
            if arr.ndim == 3 and arr.shape[0] < arr.shape[1] and arr.shape[0] < arr.shape[2]:
                arr = arr.transpose(1, 2, 0)
            elif arr.ndim == 2:
                arr = arr[:, :, np.newaxis]
            return arr
    raise KeyError(f'No RF key found in {fpath}. Keys: {list(mat_data.keys())}')

## 2. Inference on Hold-Out Phantom Data

In [ ]:
    results = {'pred_labels': [], 'true_labels': [], 'probabilities': [], 'cnr_model': [], 'cnr_baseline': [], 'filenames': []}
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    for fpath in tqdm(mat_files, desc='Inference'):
        try:
            # Use the robust extraction function
            rf_clean = extract_rf_frames(fpath)
            
            # For baseline NCC, we need pre and post
            if rf_clean.shape[2] >= 2:
                mid = rf_clean.shape[2] // 2
                rf_pre  = rf_clean[:, :, 0]
                rf_post = rf_clean[:, :, -1] # Use last for max contrast
            else:
                rf_pre  = rf_clean[:, :, 0]
                rf_post = rf_clean[:, :, 0]

            x_tensor = prepare_rf_tensor(rf_clean, PATCH_SIZE, N_FRAMES, DEVICE)

            with torch.no_grad():
                logits, disp_t = model(x_tensor)
                probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

            pred_idx  = int(probs.argmax())
            disp_np   = disp_t[0].mean(0).cpu().numpy()   # (H, W)
            cnr_m     = compute_cnr(disp_np)
            cnr_base  = baseline_ncc_cnr(rf_pre, rf_post)

            # Ground truth: check FULL PATH for labels
            full_path_lower = fpath.lower()
            if 'benign' in full_path_lower:        true_label = 0
            elif 'pre'   in full_path_lower:       true_label = 1
            elif 'malig' in full_path_lower:       true_label = 2
            else:                                  true_label = pred_idx

            results['pred_labels'].append(pred_idx)
            results['true_labels'].append(true_label)
            results['probabilities'].append(probs.tolist())
            results['cnr_model'].append(cnr_m)
            results['cnr_baseline'].append(cnr_base)
            results['filenames'].append(os.path.basename(fpath))

        except Exception as e:
            print(f'  ⚠️  Skipped {os.path.basename(fpath)}: {e}')

## 3. Statistical Estimation — MLE & 95% Confidence Intervals

We treat the **Elastographic CNR** as a random variable drawn from a normal distribution $\text{CNR} \sim \mathcal{N}(\mu, \sigma^2)$.

The **Maximum Likelihood Estimator** (MLE) for the mean is the sample mean:
$$\hat{\mu}_{\text{MLE}} = \bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i$$

The 95% **Confidence Interval** via Student's t-distribution (exact for small N):
$$\text{CI}_{95\%} = \hat{\mu} \pm t_{\alpha/2,\, N-1} \cdot \frac{s}{\sqrt{N}}$$

In [ ]:
def mle_normal_ci(
    data: np.ndarray,
    confidence: float = 0.95,
    label: str = 'variable'
) -> dict:
    """
    Compute MLE estimates (mean, variance) and confidence interval
    for normally distributed data using Student's t-distribution.

    Parameters
    ----------
    data       : 1-D array of observations
    confidence : CI level (default 0.95)
    label      : name for printing

    Returns
    -------
    dict with keys: mu_mle, sigma2_mle, ci_lower, ci_upper, se, n
    """
    n        = len(data)
    mu_mle   = np.mean(data)                  # MLE mean
    sigma2_mle = np.var(data, ddof=0)         # MLE variance (biased)
    s        = np.std(data, ddof=1)           # unbiased std (for CI)
    se       = s / np.sqrt(n)                 # standard error
    alpha    = 1 - confidence
    t_crit   = t_dist.ppf(1 - alpha / 2, df=n - 1)
    margin   = t_crit * se

    ci_lower = mu_mle - margin
    ci_upper = mu_mle + margin

    print(f'\n📐 MLE & {int(confidence*100)}% CI for {label}:')
    print(f'   N                  : {n}')
    print(f'   MLE mean (μ̂)       : {mu_mle:.4f}')
    print(f'   MLE variance (σ̂²)  : {sigma2_mle:.4f}')
    print(f'   Standard Error     : {se:.4f}')
    print(f'   t_{{α/2, {n-1}}}       : {t_crit:.4f}')
    print(f'   {int(confidence*100)}% CI           : [{ci_lower:.4f},  {ci_upper:.4f}]')

    return dict(
        mu_mle=mu_mle, sigma2_mle=sigma2_mle,
        ci_lower=ci_lower, ci_upper=ci_upper,
        se=se, t_crit=t_crit, n=n, label=label
    )


ci_model    = mle_normal_ci(cnr_model,    label='CNN-LSTM CNR')
ci_baseline = mle_normal_ci(cnr_baseline, label='Baseline NCC CNR')

# ── Normality check (Shapiro-Wilk) ────────────────────────────────────────────
if N_SAMPLES >= 3:
    w_m,  p_m  = stats.shapiro(cnr_model)
    w_b,  p_b  = stats.shapiro(cnr_baseline)
    print(f'\n🔍 Shapiro-Wilk normality test:')
    print(f'   CNN-LSTM CNR  → W={w_m:.4f}, p={p_m:.4f}  {"✅ Normal" if p_m>0.05 else "⚠️  Non-normal"}')
    print(f'   Baseline CNR  → W={w_b:.4f}, p={p_b:.4f}  {"✅ Normal" if p_b>0.05 else "⚠️  Non-normal"}')
else:
    print('⚠️  Too few samples for Shapiro-Wilk test (need ≥ 3)')

## 4. Hypothesis Testing — Wilcoxon Signed-Rank Test

### Formulation
$$H_0:\; \tilde{\mu}_{\text{model}} \leq \tilde{\mu}_{\text{baseline}} \qquad H_1:\; \tilde{\mu}_{\text{model}} > \tilde{\mu}_{\text{baseline}}$$

The **Wilcoxon signed-rank test** is used (not paired t-test) because:
- The CNR distribution may not be normally distributed (as verified by Shapiro-Wilk above)
- It is a non-parametric, distribution-free test valid for paired difference data
- It is robust to outliers inherent in phantom elastography measurements

**Decision rule:** Reject $H_0$ if $p < 0.05$

In [ ]:
# ── Wilcoxon Signed-Rank Test ─────────────────────────────────────────────────
print('=' * 65)
print('  WILCOXON SIGNED-RANK HYPOTHESIS TEST')
print('=' * 65)
print(f'  H₀ : median(CNR_model) ≤ median(CNR_baseline)')
print(f'  H₁ : median(CNR_model) >  median(CNR_baseline)  [one-sided]')
print(f'  α  = 0.05')
print()

differences = cnr_model - cnr_baseline

if N_SAMPLES < 5:
    print('⚠️  Too few samples for reliable Wilcoxon test (need ≥ 5). Skipping.')
    stat, p_value, effect_size = np.nan, np.nan, np.nan
else:
    stat, p_value = wilcoxon(
        cnr_model, cnr_baseline,
        alternative='greater',      # one-sided: model > baseline
        zero_method='wilcox',
        correction=True             # continuity correction for finite samples
    )

    # Effect size: rank-biserial correlation r = 1 - (2·W) / (n·(n+1)/2)
    n_pairs    = N_SAMPLES
    max_w      = n_pairs * (n_pairs + 1) / 2
    effect_size = 1 - (2 * stat) / max_w   # r ∈ [-1, 1]; >0 means model wins

    print(f'  Test statistic W  : {stat:.2f}')
    print(f'  p-value           : {p_value:.6f}')
    print(f'  Effect size (r)   : {effect_size:.4f}  (>0.3 = moderate, >0.5 = large)')
    print()

    alpha = 0.05
    if p_value < alpha:
        print(f'  ✅ RESULT: Reject H₀  (p={p_value:.4f} < α={alpha})')
        print(f'  The CNN-LSTM model CNR is statistically significantly higher')
        print(f'  than the baseline NCC algorithm.  Model is validated for production.')
    else:
        print(f'  ❌ RESULT: Fail to reject H₀  (p={p_value:.4f} ≥ α={alpha})')
        print(f'  Insufficient evidence. Model requires further training.')

print('=' * 65)

# Save results
stats_results = {
    'n_samples':    int(N_SAMPLES),
    'cnr_model':    {'mean': float(cnr_model.mean()), 'std': float(cnr_model.std()),
                     'ci_lower': ci_model['ci_lower'],    'ci_upper': ci_model['ci_upper']},
    'cnr_baseline': {'mean': float(cnr_baseline.mean()), 'std': float(cnr_baseline.std()),
                     'ci_lower': ci_baseline['ci_lower'], 'ci_upper': ci_baseline['ci_upper']},
    'wilcoxon': {
        'statistic':  float(stat) if not np.isnan(stat) else None,
        'p_value':    float(p_value) if not np.isnan(p_value) else None,
        'effect_size': float(effect_size) if not np.isnan(effect_size) else None,
        'reject_h0':  bool(p_value < 0.05) if not np.isnan(p_value) else None,
    }
}
with open(os.path.join(OUT_DIR, 'statistical_results.json'), 'w') as f:
    json.dump(stats_results, f, indent=2)
print(f'\n💾 Statistical results saved to {OUT_DIR}statistical_results.json')

## 5. Visualisation — CNR Distributions & CI Plot

In [ ]:
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35)

# ── Panel A: CNR Distributions ────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ax_a.hist(cnr_model,    bins=15, alpha=0.65, color='steelblue', label='CNN-LSTM', edgecolor='white')
ax_a.hist(cnr_baseline, bins=15, alpha=0.65, color='tomato',    label='Baseline NCC', edgecolor='white')
ax_a.axvline(cnr_model.mean(),    color='steelblue', ls='--', lw=2, label=f'Model μ={cnr_model.mean():.2f}')
ax_a.axvline(cnr_baseline.mean(), color='tomato',    ls='--', lw=2, label=f'Baseline μ={cnr_baseline.mean():.2f}')
ax_a.set_xlabel('Elastographic CNR', fontsize=11)
ax_a.set_ylabel('Frequency', fontsize=11)
ax_a.set_title('A. CNR Distributions', fontweight='bold')
ax_a.legend(fontsize=8)

# ── Panel B: 95% Confidence Intervals ────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
labels_ci = ['CNN-LSTM\nModel', 'Baseline\nNCC']
means_ci  = [ci_model['mu_mle'],    ci_baseline['mu_mle']]
lo_err    = [ci_model['mu_mle']   - ci_model['ci_lower'],
             ci_baseline['mu_mle'] - ci_baseline['ci_lower']]
hi_err    = [ci_model['ci_upper']   - ci_model['mu_mle'],
             ci_baseline['ci_upper'] - ci_baseline['mu_mle']]
colors_ci = ['steelblue', 'tomato']
bars = ax_b.bar(labels_ci, means_ci, color=colors_ci, alpha=0.75, width=0.4,
                yerr=[lo_err, hi_err], capsize=10, error_kw=dict(elinewidth=2, ecolor='black'))
ax_b.set_ylabel('Mean CNR', fontsize=11)
ax_b.set_title('B. MLE Means with 95% CI', fontweight='bold')
for bar, m in zip(bars, means_ci):
    ax_b.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
              f'{m:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# ── Panel C: Paired Differences (Bland-Altman style) ─────────────────────────
ax_c = fig.add_subplot(gs[0, 2])
ax_c.scatter(range(N_SAMPLES), differences, alpha=0.6, c='purple', s=18, zorder=3)
ax_c.axhline(0,              color='black', lw=1.5, ls='-')
ax_c.axhline(differences.mean(), color='purple', lw=2, ls='--',
             label=f'Mean diff = {differences.mean():.3f}')
ax_c.fill_between(range(N_SAMPLES),
                  differences.mean() - 1.96*differences.std(),
                  differences.mean() + 1.96*differences.std(),
                  alpha=0.15, color='purple', label='±1.96 SD')
ax_c.set_xlabel('Sample index', fontsize=10)
ax_c.set_ylabel('ΔCNR (Model − Baseline)', fontsize=10)
ax_c.set_title('C. Paired CNR Differences', fontweight='bold')
ax_c.legend(fontsize=8)

# ── Panel D: Wilcoxon p-value summary ────────────────────────────────────────
ax_d = fig.add_subplot(gs[1, 0])
ax_d.axis('off')
summary_text = (
    'WILCOXON SIGNED-RANK TEST\n'
    '─────────────────────────────────────\n'
    f'N samples       : {N_SAMPLES}\n'
    f'W statistic     : {stat:.2f}\n'
    f'p-value         : {p_value:.6f}\n'
    f'Effect size (r) : {effect_size:.4f}\n'
    '─────────────────────────────────────\n'
    + (f'✅ Reject H₀ (p < 0.05)' if not np.isnan(p_value) and p_value < 0.05
       else '❌ Fail to reject H₀')
)
ax_d.text(0.05, 0.95, summary_text, transform=ax_d.transAxes,
          fontsize=10, verticalalignment='top', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# ── Panel E: MLE Parameter Uncertainty (Normal PDF overlay) ──────────────────
ax_e = fig.add_subplot(gs[1, 1])
x_range = np.linspace(
    min(cnr_model.min(), cnr_baseline.min()) - 0.5,
    max(cnr_model.max(), cnr_baseline.max()) + 0.5, 300
)
for data, col, ci_d, lbl in [
    (cnr_model,    'steelblue', ci_model,    'CNN-LSTM'),
    (cnr_baseline, 'tomato',    ci_baseline, 'Baseline')
]:
    pdf = norm.pdf(x_range, loc=ci_d['mu_mle'], scale=np.sqrt(ci_d['sigma2_mle']))
    ax_e.plot(x_range, pdf, color=col, lw=2, label=lbl)
    ax_e.axvspan(ci_d['ci_lower'], ci_d['ci_upper'], alpha=0.12, color=col)
ax_e.set_xlabel('CNR', fontsize=11)
ax_e.set_ylabel('Probability Density', fontsize=11)
ax_e.set_title('E. MLE Normal PDFs (shaded = 95% CI)', fontweight='bold')
ax_e.legend(fontsize=9)

# ── Panel F: Box-plot comparison ─────────────────────────────────────────────
ax_f = fig.add_subplot(gs[1, 2])
bplot = ax_f.boxplot([cnr_model, cnr_baseline], labels=['CNN-LSTM\nModel', 'Baseline\nNCC'],
                     patch_artist=True, notch=True, bootstrap=1000)
for patch, col in zip(bplot['boxes'], ['steelblue', 'tomato']):
    patch.set_facecolor(col); patch.set_alpha(0.6)
ax_f.set_ylabel('Elastographic CNR', fontsize=11)
ax_f.set_title('F. CNR Box Plots (notch = 95% CI on median)', fontweight='bold')

p_label = f'p = {p_value:.4f}' if not np.isnan(p_value) else 'p = N/A'
sig_str  = '***' if not np.isnan(p_value) and p_value < 0.001 else \
           '**'  if not np.isnan(p_value) and p_value < 0.01  else \
           '*'   if not np.isnan(p_value) and p_value < 0.05  else 'ns'
y_top = max(cnr_model.max(), cnr_baseline.max()) * 1.08
ax_f.annotate('', xy=(2, y_top), xytext=(1, y_top),
              arrowprops=dict(arrowstyle='-', color='black', lw=1.5))
ax_f.text(1.5, y_top * 1.01, f'{sig_str} ({p_label})', ha='center', fontsize=10)

plt.suptitle('Phase 3: Clinical Statistical Validation — CNN-LSTM vs Baseline NCC',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig(os.path.join(OUT_DIR, 'fig_04_statistical_validation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Statistical validation figure saved')

## 6. Classification Performance Metrics

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm_arr = confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
disp = ConfusionMatrixDisplay(cm_arr, display_labels=CLASS_NAMES)
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold')

# Normalised (recall per class)
cm_norm = cm_arr.astype(float) / (cm_arr.sum(axis=1, keepdims=True) + 1e-8)
disp2   = ConfusionMatrixDisplay(cm_norm.round(2), display_labels=CLASS_NAMES)
disp2.plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Confusion Matrix (Normalised — Recall)', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fig_05_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Classification Report ─────────────────────────────────────────────────────
print('\n📋 Classification Report')
print('─' * 55)
print(classification_report(true_labels, pred_labels, target_names=CLASS_NAMES, zero_division=0))

# ── Per-class accuracy ────────────────────────────────────────────────────────
per_class_acc = cm_arr.diagonal() / (cm_arr.sum(axis=1) + 1e-8)
print('Per-class Accuracy:')
for cls, acc in zip(CLASS_NAMES, per_class_acc):
    print(f'   {cls:15s}: {acc:.3f}  ({acc*100:.1f}%)')
overall_acc = (pred_labels == true_labels).mean()
print(f'   Overall accuracy : {overall_acc:.3f}  ({overall_acc*100:.1f}%)')

## 7. Multi-Class ROC-AUC Analysis

In [ ]:
if probs_matrix.shape[0] > 0 and len(np.unique(true_labels)) > 1:
    # Binarise true labels for OvR ROC
    true_bin = label_binarize(true_labels, classes=[0, 1, 2])   # (N, 3)

    fig, ax = plt.subplots(figsize=(7, 6))
    colors_roc = ['steelblue', 'darkorange', 'forestgreen']
    aucs = []

    for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors_roc)):
        if true_bin[:, i].sum() == 0:
            print(f'   Skipping {cls} ROC — no positive samples')
            continue
        fpr, tpr, _ = roc_curve(true_bin[:, i], probs_matrix[:, i])
        auc_val     = roc_auc_score(true_bin[:, i], probs_matrix[:, i])
        aucs.append(auc_val)
        ax.plot(fpr, tpr, color=col, lw=2.5,
                label=f'{cls} (AUC = {auc_val:.3f})')

    ax.plot([0,1],[0,1],'k--', lw=1.5, label='Random (AUC = 0.500)')
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('Multi-Class ROC Curves (One-vs-Rest)', fontweight='bold', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, 'fig_06_roc_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

    if aucs:
        print(f'\n   Macro-averaged AUC: {np.mean(aucs):.4f}')
else:
    print('⚠️  Insufficient class diversity for ROC analysis.')

## 8. Final Validation Summary

In [ ]:
print('\n' + '═'*65)
print('  PHASE 3 — CLINICAL VALIDATION SUMMARY')
print('═'*65)
print(f'  Dataset          : Hold-out phantom/in-vivo (Dataset 2, 20%)')
print(f'  N samples        : {N_SAMPLES}')
print()
print(f'  ── CNR Performance ─────────────────────────────────────')
print(f'  CNN-LSTM CNR     : {cnr_model.mean():.3f} ± {cnr_model.std():.3f}')
print(f'  95% CI           : [{ci_model["ci_lower"]:.3f}, {ci_model["ci_upper"]:.3f}]')
print(f'  Baseline NCC CNR : {cnr_baseline.mean():.3f} ± {cnr_baseline.std():.3f}')
print(f'  95% CI           : [{ci_baseline["ci_lower"]:.3f}, {ci_baseline["ci_upper"]:.3f}]')
print(f'  CNR Improvement  : {((cnr_model.mean()-cnr_baseline.mean())/cnr_baseline.mean()*100):+.1f}%')
print()
print(f'  ── Wilcoxon Hypothesis Test ─────────────────────────────')
print(f'  H₀: model CNR ≤ baseline CNR (one-sided, α=0.05)')
print(f'  W  = {stat:.2f}  |  p = {p_value:.6f}  |  r = {effect_size:.4f}')
decision = '✅ VALIDATED — Reject H₀' if not np.isnan(p_value) and p_value < 0.05 \
           else '❌ NOT VALIDATED — Fail to reject H₀'
print(f'  Decision         : {decision}')
print()
print(f'  ── Classification ───────────────────────────────────────')
print(f'  Overall accuracy : {overall_acc*100:.1f}%')
for cls, acc in zip(CLASS_NAMES, per_class_acc):
    print(f'  {cls:15s} accuracy: {acc*100:.1f}%')
print('═'*65)

# ── Save final summary JSON ───────────────────────────────────────────────────
summary = {
    **stats_results,
    'classification': {
        'overall_accuracy':  float(overall_acc),
        'per_class_accuracy': {cls: float(acc) for cls, acc in zip(CLASS_NAMES, per_class_acc)},
        'confusion_matrix':   cm_arr.tolist(),
    }
}
with open(os.path.join(OUT_DIR, 'phase3_summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\n💾 Full summary saved → {OUT_DIR}phase3_summary.json')
print('✅ Phase 3 Statistical Validation complete')